In [4]:
!pip install selenium webdriver-manager

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\USER\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

In [6]:
import re


def get_driver():
    """Return a configured Selenium WebDriver instance."""
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    return driver


def get_status(card, modifier):
    """Get text from a card-product__status span by its modifier class."""
    el = card.find("span", class_=lambda c: c and "card-product__status" in c and modifier in c)
    return el.get_text(strip=True) if el else None


def parse_car_cards(soup):
    """Extract car fields from all card-product elements on a page."""
    cars = []
    cards = soup.find_all("div", class_="card-product")

    for card in cards:
        car = {}

        # Make, Model, Year are packed into one span:
        # e.g. "2010/5 TOYOTA HIACE WAGON GRAND CABIN"
        title_el = card.find("span", class_="card-product__product")
        if title_el:
            title = title_el.get_text(strip=True)
            # Extract year: leading "YYYY" or "YYYY/MM"
            year_match = re.match(r"^(\d{4})(?:/\d+)?\s+", title)
            if year_match:
                car["Year"] = year_match.group(1)
                rest = title[year_match.end():].strip()
            else:
                car["Year"] = None
                rest = title
            # First word after year = Make, remainder = Model
            parts = rest.split(" ", 1)
            car["Make"] = parts[0] if parts else None
            car["Model"] = parts[1] if len(parts) > 1 else None
        else:
            car["Make"] = car["Model"] = car["Year"] = None

        # Mileage  e.g. "281,000km"
        car["Mileage"] = get_status(card, "-mileage")

        # Engine size  e.g. "2,693cc"
        car["Engine_Size"] = get_status(card, "-engine-capacity")

        # Fuel type  e.g. "PETROL"
        car["Fuel_Type"] = get_status(card, "-fuel-type")

        # Transmission  e.g. "AT"
        car["Transmission"] = get_status(card, "-transmission")

        # Vehicle price  e.g. "10,330" (USD)
        price_area = card.find("div", class_="card-product__vehicle-price")
        if price_area:
            price_el = price_area.find("span", class_="card-product__price")
            car["Price"] = price_el.get_text(strip=True) if price_el else None
        else:
            car["Price"] = None

        # Body type is not shown on the card listing;
        # pass body_type= in the search URL to filter and tag rows.
        car["Body_Type"] = None

        cars.append(car)
    return cars


def scrape_sbt_japan(max_pages=5,
                     base_url="https://www.sbtjapan.com/used-cars/search",
                     body_type=None):
                     
    """Scrape car listings from SBT Japan over multiple pages."""
    driver = get_driver()
    all_cars = []

    try:
        for page in range(1, max_pages + 1):
            url = f"{base_url}?page={page}"
            print(f"Scraping page {page}: {url}")
            driver.get(url)

            # Wait for car cards to load
            try:
                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.CLASS_NAME, "card-product"))
                )
            except Exception:
                print(f"  No car cards found on page {page}, stopping.")
                break

            # Scroll to trigger lazy-loaded images/content
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

            soup = BeautifulSoup(driver.page_source, "lxml")
            cars = parse_car_cards(soup)

            if not cars:
                print(f"  No cars parsed on page {page}, stopping.")
                break

            # Tag body type if caller specified one
            if body_type:
                for c in cars:
                    c["Body Type"] = body_type

            print(f"  Found {len(cars)} cars on page {page}")
            all_cars.extend(cars)
            time.sleep(1.5)  # polite delay
    finally:
        driver.quit()

    df = pd.DataFrame(all_cars, columns=[
        "Make", "Model", "Year", "Mileage",
        "Engine_Size", "Fuel_Type", "Transmission", "Body_Type", "Price"
    ])
    return df


In [7]:
# Scrape 5 pages (~100-150 cars). Increase max_pages for more data.
# Filter examples:
#   base_url="https://www.sbtjapan.com/used-cars/search?make_id=2"  -> Toyota only
#   base_url="https://www.sbtjapan.com/used-cars/search?body_type=SUV"
df = scrape_sbt_japan(max_pages=10)

print(f"\nTotal cars scraped: {len(df)}")
print(df.head(10))

# Save to CSV
df.to_csv("sbt_japan_cars.csv", index=False)
print("\nSaved to sbt_japan_cars.csv")

Scraping page 1: https://www.sbtjapan.com/used-cars/search?page=1
  Found 50 cars on page 1
Scraping page 2: https://www.sbtjapan.com/used-cars/search?page=2
  Found 50 cars on page 2
Scraping page 3: https://www.sbtjapan.com/used-cars/search?page=3
  Found 50 cars on page 3
Scraping page 4: https://www.sbtjapan.com/used-cars/search?page=4
  Found 50 cars on page 4
Scraping page 5: https://www.sbtjapan.com/used-cars/search?page=5
  Found 50 cars on page 5
Scraping page 6: https://www.sbtjapan.com/used-cars/search?page=6
  Found 50 cars on page 6
Scraping page 7: https://www.sbtjapan.com/used-cars/search?page=7
  Found 50 cars on page 7
Scraping page 8: https://www.sbtjapan.com/used-cars/search?page=8
  Found 50 cars on page 8
Scraping page 9: https://www.sbtjapan.com/used-cars/search?page=9
  Found 50 cars on page 9
Scraping page 10: https://www.sbtjapan.com/used-cars/search?page=10
  Found 50 cars on page 10

Total cars scraped: 500
       Make                                         

In [8]:
df = pd.read_csv('sbt_japan_cars.csv')
df

,Make,Model,Year,Mileage,Engine_Size,Fuel_Type,Transmission,Body_Type,Price
0,NISSAN,ATLAS TRUCK 1.5t FLAT BODY,2018,"206,000km","3,000cc",DIESEL,MT,NaN,"6,150"
1,NISSAN,ATLAS TRUCK 1.35t BOX,2019,"222,000km","2,000cc",PETROL,AT,NaN,"5,850"
2,TOYOTA,TOYOACE TRUCK 1.5t FLAT BODY,2016,"328,000km","3,000cc",DIESEL,MT,NaN,"7,050"
3,ISUZU,ELF TRUCK 2.0t DUMP,1997,"193,000km","4,334cc",DIESEL,MT,NaN,"9,090"
4,NISSAN,XTRAIL 20XT,2009,"122,000km","2,000cc",PETROL,AT,NaN,"2,420"
...,...,...,...,...,...,...,...,...,...
495,TOYOTA,COROLLA AXIO HYBRID,2019,"108,000km","1,500cc",HYBRID(PETROL),AT,NaN,"6,200"
496,TOYOTA,COROLLA AXIO HYBRID EX,2021,"108,000km","1,500cc",HYBRID(PETROL),AT,NaN,"7,690"
497,TOYOTA,COROLLA AXIO HYBRID,2019,"158,000km","1,500cc",HYBRID(PETROL),AT,NaN,"5,380"
498,TOYOTA,COROLLA AXIO HYBRID,2019,"136,000km","1,500cc",HYBRID(PETROL),AT,NaN,"5,880"


# EDA - Exploratory Data Analysis

In [9]:
# Explore the data
df.columns

Index(['Make', 'Model', 'Year', 'Mileage', 'Engine_Size', 'Fuel_Type',
       'Transmission', 'Body_Type', 'Price'],
      dtype='str')

In [10]:
df.info() 

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Make          500 non-null    str    
 1   Model         500 non-null    str    
 2   Year          500 non-null    int64  
 3   Mileage       500 non-null    str    
 4   Engine_Size   500 non-null    str    
 5   Fuel_Type     500 non-null    str    
 6   Transmission  500 non-null    str    
 7   Body_Type     0 non-null      float64
 8   Price         500 non-null    str    
dtypes: float64(1), int64(1), str(7)
memory usage: 35.3 KB


In [11]:
df.head()

,Make,Model,Year,Mileage,Engine_Size,Fuel_Type,Transmission,Body_Type,Price
0,NISSAN,ATLAS TRUCK 1.5t FLAT BODY,2018,"206,000km","3,000cc",DIESEL,MT,NaN,"6,150"
1,NISSAN,ATLAS TRUCK 1.35t BOX,2019,"222,000km","2,000cc",PETROL,AT,NaN,"5,850"
2,TOYOTA,TOYOACE TRUCK 1.5t FLAT BODY,2016,"328,000km","3,000cc",DIESEL,MT,NaN,"7,050"
3,ISUZU,ELF TRUCK 2.0t DUMP,1997,"193,000km","4,334cc",DIESEL,MT,NaN,"9,090"
4,NISSAN,XTRAIL 20XT,2009,"122,000km","2,000cc",PETROL,AT,NaN,"2,420"


In [12]:
df['Mileage'] = (df['Mileage'].str.replace('km','').str.replace(',', '').astype(int)) # Convert Mileage: remove 'km' and commas

In [13]:
df.head(3)

,Make,Model,Year,Mileage,Engine_Size,Fuel_Type,Transmission,Body_Type,Price
0,NISSAN,ATLAS TRUCK 1.5t FLAT BODY,2018,206000,"3,000cc",DIESEL,MT,NaN,"6,150"
1,NISSAN,ATLAS TRUCK 1.35t BOX,2019,222000,"2,000cc",PETROL,AT,NaN,"5,850"
2,TOYOTA,TOYOACE TRUCK 1.5t FLAT BODY,2016,328000,"3,000cc",DIESEL,MT,NaN,"7,050"


In [14]:
df['Engine_Size'] = (df['Engine_Size'].str.replace('cc','').str.replace(',', '').astype(int)) # Convert Mileage: remove 'cc' and commas

In [15]:
# Convert Price: replace 'Ask' with NaN, remove commas, then convert

df['Price'] = df['Price'].replace('Ask', pd.NA)

df['Price'] = (df['Price'].str.replace(',', '').astype(float))

In [16]:
df.describe()

,Year,Mileage,Engine_Size,Body_Type,Price
count,500.000000,500.000000,500.00000,0.0,500.00000
mean,2013.150000,134904.000000,1706.98200,NaN,4376.60000
std,5.174655,71830.676428,764.39343,NaN,4177.52648
min,1994.000000,10000.000000,0.00000,NaN,650.00000
25%,2010.000000,93750.000000,1300.00000,NaN,2050.00000
50%,2013.000000,123000.000000,1500.00000,NaN,3065.00000
75%,2017.000000,164250.000000,2000.00000,NaN,5312.50000
max,2023.000000,860000.000000,4890.00000,NaN,30680.00000


In [17]:
df.dtypes

Make                str
Model               str
Year              int64
Mileage           int64
Engine_Size       int64
Fuel_Type           str
Transmission        str
Body_Type       float64
Price           float64
dtype: object

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Make          500 non-null    str    
 1   Model         500 non-null    str    
 2   Year          500 non-null    int64  
 3   Mileage       500 non-null    int64  
 4   Engine_Size   500 non-null    int64  
 5   Fuel_Type     500 non-null    str    
 6   Transmission  500 non-null    str    
 7   Body_Type     0 non-null      float64
 8   Price         500 non-null    float64
dtypes: float64(2), int64(3), str(4)
memory usage: 35.3 KB


In [35]:
# Create a combined vehicle_model column by concatenating Make and Model
df['Vehicle_Model'] = (df['Make'].astype(str).str.strip() + ' ' + df['Model'].astype(str).str.strip()).str.replace('\\s+', ' ', regex=True).str.strip()

In [34]:
# Ensure both Vehicle_Model and vehicle_model columns are removed
for col in ['Vehicle_Model','vehicle_model']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# Verify Make/Model and current columns
print(list(df.columns))

['Make', 'Model', 'Year', 'Mileage', 'Engine_Size', 'Fuel_Type', 'Transmission', 'Body_Type', 'Price']


In [36]:
# Drop the combined vehicle_model column (if present)
if 'vehicle_model' in df.columns:
    df.drop(columns=['vehicle_model'], inplace=True)

# Show Make/Model and current columns to verify
df.columns

Index(['Make', 'Model', 'Year', 'Mileage', 'Engine_Size', 'Fuel_Type',
       'Transmission', 'Body_Type', 'Price', 'Vehicle_Model'],
      dtype='str')

In [37]:
# Features (X)
X = df[['Make', 'Model', 'Vehicle_Model', 'Body_Type', 'Year', 'Mileage', 'Engine_Size', 'Fuel_Type', 'Transmission']]

# Target (y)
y = df['Price']

print(X.head())
print(y.head())

Empty DataFrame
Columns: [Make, Model, Vehicle_Model, Body_Type, Year, Mileage, Engine_Size, Fuel_Type, Transmission]
Index: []
Series([], Name: Price, dtype: float64)


In [38]:
print(y.head())

Series([], Name: Price, dtype: float64)


In [39]:
df.isnull().sum() # Check null values

Make             0
Model            0
Year             0
Mileage          0
Engine_Size      0
Fuel_Type        0
Transmission     0
Body_Type        0
Price            0
Vehicle_Model    0
dtype: int64

In [40]:
# Drop rows with Null price values
df.dropna(subset=['Price', 'Body_Type'], inplace=True)

df.isnull().sum()

Make             0
Model            0
Year             0
Mileage          0
Engine_Size      0
Fuel_Type        0
Transmission     0
Body_Type        0
Price            0
Vehicle_Model    0
dtype: int64

In [41]:
# Split data into Test data and Training data

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [42]:
df

,Make,Model,Year,Mileage,Engine_Size,Fuel_Type,Transmission,Body_Type,Price,Vehicle_Model


# Storing the extracted data in a database

In [43]:
# Storing the extracted data in a database

import sqlalchemy
import psycopg2

In [44]:
from sqlalchemy import create_engine, text

In [45]:
# Clean version of the dataset

df.to_csv('sbt_japan_cars_cleaned.csv', index=False)

In [57]:
# Connecting to pgsql
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = "sbt_japan_cars"

engine_psql = create_engine(f"postgresql+psycopg2://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

try:
    with engine_psql.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Connection to PostgreSQL OK")
except Exception as e:
    print(f"Connection to PostgreSQL failed: {e}")

Connection to PostgreSQL OK


In [58]:
# Create a database

db_name = 'sbt_japan_cars'

with engine_psql.connect() as conn:

    # Enable autocommit
    conn = conn.execution_options(isolation_level="AUTOCOMMIT")

    try:
        conn.execute(text(f"CREATE DATABASE {db_name}"))
        print(f"Database '{db_name}' created successfully!")
        
    except Exception as e:
        if "already exists" in str(e):
            print(f"Database '{db_name}' already exists, skipping")
        else:
            print("Database creation failed: {e}")

Database 'sbt_japan_cars' already exists, skipping


In [60]:
table_name = 'sbt_japan_cars'

# Step 1: Create the table explicitly with defined column types

with engine_psql.connect() as conn:

    conn = conn.execution_options(isolation_level="AUTOCOMMIT")

    try:
        conn.execute(text(f"""
            CREATE TABLE IF NOT EXISTS {table_name} (
                "Make"         VARCHAR(100),
                "Model"        VARCHAR(200),
                "Year"         INTEGER,
                "Mileage"      INTEGER,
                "Engine_Size"  INTEGER,
                "Fuel_Type"    VARCHAR(50),
                "Transmission" VARCHAR(10),
                "Price"        FLOAT
            )
        """))

        print(f"Table '{table_name}' created successfully!")

    except Exception as e:
        if "already exists" in str(e):
            print(f"Table '{table_name}' already exists, skipping")
        else:
            print(f"Table creation failed: {e}")

Table 'sbt_japan_cars' created successfully!


In [61]:
# Step 2: Export the DataFrame into the table

try:
    df.to_sql(name='sbt_japan_cars', con=engine_psql, if_exists='replace', index=False)

    print(f"Data exported to '{table_name}' successfully!")
    
except Exception as e:
    if "already exists" in str(e):
        print(f"Table '{table_name}' already exists, skipping export")
    else:
        print(f"Failed to export data: {e}")

Data exported to 'sbt_japan_cars' successfully!
